[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_19/2_sql.ipynb)

# Day 19: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (31 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | The most loyal users | medium | 7 |
| Q2 | Who is on a streak right now? | medium | 6 |
| Q3 | Rating sessions | medium | 8 |
| Q4 | Long breaks and comebacks (review) | medium | 5 |
| Q5 | How much activity comes from streaks? (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: MovieLens small (GroupLens, research and education use).
def movielens(short):
    path = os.path.join(DATA, f"movielens_{short}.csv")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading MovieLens")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip").read()))
        for s in ("ratings", "movies", "tags", "links"):
            with open(os.path.join(DATA, f"movielens_{s}.csv"), "wb") as f:
                f.write(z.read(f"ml-latest-small/{s}.csv"))
    return pd.read_csv(path)

_r = movielens("ratings")
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]] \
    .to_sql("ml_ratings", db, index=False)
movielens("movies").rename(columns={"movieId": "movie_id"}).to_sql("ml_movies", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title` (with the year), `genres` (several genres joined by a vertical bar, e.g. Comedy\|Romance) |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. The most loyal users

*medium, about 7 min*

Shopping app (made up). A **streak** is a run of consecutive calendar days on which a user has a
`daily_activity` row. For every user find their **longest streak** (ties: the earliest one). Return the **5 users**
with the longest streaks: `user_id`, `start_date`, `end_date`, `longest_streak` (days), longest first, then user_id.

Example: active on Jan 3, 4, 5, 7, 8 gives streaks of 3 (Jan 3 to 5) and 2 (Jan 7 to 8); the longest is 3.

Follow-up: what if a user can have two rows on the same day?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "consecutive days" = gaps and islands. Pattern: date minus ROW_NUMBER is constant inside a run of consecutive dates; group by that difference.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE grp: `julianday(activity_date) - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date)`.
2. CTE streaks: group by user and that value: MIN, MAX, COUNT.
3. ROW_NUMBER per user by days DESC, start_date; keep 1; order and LIMIT 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH grp AS (
    SELECT user_id, activity_date,
           julianday(activity_date)
             - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) AS island
    FROM daily_activity
), streaks AS (
    SELECT user_id, MIN(activity_date) AS start_date, MAX(activity_date) AS end_date, COUNT(*) AS days
    FROM grp
    GROUP BY user_id, island
), ranked AS (
    SELECT user_id, start_date, end_date, days,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY days DESC, start_date) AS rn
    FROM streaks
)
SELECT user_id, start_date, end_date, days AS longest_streak
FROM ranked
WHERE rn = 1
ORDER BY longest_streak DESC, user_id
LIMIT 5
```

Result:

| user_id | start_date | end_date | longest_streak |
|---|---|---|---|
| 259 | 2024-01-11 | 2024-03-12 | 62 |
| 166 | 2024-01-27 | 2024-03-27 | 61 |
| 698 | 2024-01-28 | 2024-03-21 | 54 |
| 935 | 2024-02-07 | 2024-03-31 | 54 |
| 522 | 2024-02-11 | 2024-03-30 | 49 |

**Why:** Within a run of consecutive dates, both the date and the row number go up by 1 each row, so their difference stays the same; a gap in dates increases the difference and starts a new island. Example: Jan 3, 4, 5, 7, 8 with row numbers 1 to 5 give differences Jan 2, Jan 2, Jan 2, Jan 3, Jan 3. The top user was active 62 days in a row. Follow-up: duplicates per day break the trick (the row number grows but the date does not). Deduplicate first (`SELECT DISTINCT user_id, activity_date`) or use DENSE_RANK instead of ROW_NUMBER. PostgreSQL: `activity_date - ROW_NUMBER() OVER (...)::int` gives a date.

**Complexity:** One window sort per user, then two group bys; linear after sorting.

**Common mistakes:** Partitioning ROW_NUMBER by something other than the user. Forgetting duplicates per day. Using LAG(date) = date - 1 alone (finds the joints, but you still need a running sum to label the runs).

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q2. Who is on a streak right now?

*medium, about 6 min*

Shopping app. The **current streak** of a user is the streak that includes the **last date in the
data** (2024-03-31; do not hard-code it). Per `platform` return `active_on_last_day`, `streak_7plus` (users whose
current streak is at least 7 days), `avg_current_streak` (1 decimal) and `max_current`.

Example: a user active Mar 29, 30, 31 has a current streak of 3; a user last active on Mar 30 has none.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: same islands as Q1, then keep only the island whose end date equals the overall last date.

</details>

<details><summary><b>Hint 2</b></summary>

1. Reuse the grp / streaks CTEs.
2. `WHERE end_date = (SELECT MAX(activity_date) FROM daily_activity)`.
3. Join app_users for platform; conditional SUM for 7+.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH grp AS (
    SELECT user_id, activity_date,
           julianday(activity_date)
             - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) AS island
    FROM daily_activity
), streaks AS (
    SELECT user_id, MIN(activity_date) AS start_date, MAX(activity_date) AS end_date, COUNT(*) AS days
    FROM grp
    GROUP BY user_id, island
)
SELECT u.platform,
       COUNT(*)                                     AS active_on_last_day,
       SUM(CASE WHEN s.days >= 7 THEN 1 ELSE 0 END) AS streak_7plus,
       ROUND(AVG(s.days), 1)                        AS avg_current_streak,
       MAX(s.days)                                  AS max_current
FROM streaks s
JOIN app_users u ON u.user_id = s.user_id
WHERE s.end_date = (SELECT MAX(activity_date) FROM daily_activity)
GROUP BY u.platform
ORDER BY u.platform
```

Result:

| platform | active_on_last_day | streak_7plus | avg_current_streak | max_current |
|---|---|---|---|---|
| android | 53 | 14 | 6.3 | 54 |
| ios | 51 | 20 | 6.1 | 21 |
| web | 11 | 4 | 11.0 | 49 |

**Why:** Each user has at most one island that ends on the last date, so the filter returns one row per user active that day (115 users, the DAU of March 31). A scalar subquery for the last date keeps the query correct when new data arrives. Web has only 11 users here, so its average (11.0) is driven by one long streak: always show the count next to an average.

**Complexity:** Same as Q1 plus a small join.

**Common mistakes:** Hard-coding the date. Using `end_date >= today - 1` and counting users whose streak just broke. Reporting the web average without its tiny base.

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-subqueries](https://mahsa-agz.github.io/ds-handbook/#sql-subqueries)

</details>

---
### Q3. Rating sessions

*medium, about 8 min*

MovieLens (real). A **rating session** of a user is a run of ratings where each rating comes at most
**30 minutes** after the previous one of the same user; a longer gap (or the first rating) starts a new session.

Return one summary row: `users`, `sessions`, `avg_ratings_per_session` (1 decimal), `single_rating_sessions` and
`biggest_session` (most ratings in one session).

Example: a user rates at 10:00, 10:10, 10:50, 10:55: two sessions (2 and 2 ratings), because 10:10 to 10:50 is 40
minutes.

Follow-up: `biggest_session` is over 1,000 ratings. Is that a real human session?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "a new group starts when the gap is larger than X". Pattern: islands by flag + running sum: LAG to compute the gap, flag 1 when a session starts, cumulative SUM of the flag = session number.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE r: `new_session = 1` when LAG(rated_at) is NULL or the gap in minutes is > 30 (`(julianday(a) - julianday(b)) * 24 * 60`).
2. CTE s: `SUM(new_session) OVER (PARTITION BY user_id ORDER BY rated_at ROWS UNBOUNDED PRECEDING)`.
3. Group by user and session number, then summarise.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH r AS (
    SELECT user_id, rated_at,
           CASE WHEN LAG(rated_at) OVER w IS NULL
                  OR (julianday(rated_at) - julianday(LAG(rated_at) OVER w)) * 24 * 60 > 30
                THEN 1 ELSE 0 END AS new_session
    FROM ml_ratings
    WINDOW w AS (PARTITION BY user_id ORDER BY rated_at)
), s AS (
    SELECT user_id, rated_at,
           SUM(new_session) OVER (PARTITION BY user_id ORDER BY rated_at ROWS UNBOUNDED PRECEDING) AS session_no
    FROM r
), per_session AS (
    SELECT user_id, session_no, COUNT(*) AS ratings
    FROM s
    GROUP BY user_id, session_no
)
SELECT COUNT(DISTINCT user_id)                              AS users,
       COUNT(*)                                             AS sessions,
       ROUND(AVG(ratings), 1)                               AS avg_ratings_per_session,
       SUM(CASE WHEN ratings = 1 THEN 1 ELSE 0 END)         AS single_rating_sessions,
       MAX(ratings)                                         AS biggest_session
FROM per_session
```

Result:

| users | sessions | avg_ratings_per_session | single_rating_sessions | biggest_session |
|---|---|---|---|---|
| 610 | 7145 | 14.1 | 3235 | 1019 |

**Why:** When the islands are defined by a threshold on gaps (not by consecutive integers), the date minus row number trick does not work; the flag plus running sum does, and it is the standard sessionization query. Say `ROWS UNBOUNDED PRECEDING` explicitly: with the default RANGE frame, rows with the same timestamp share one running sum (harmless here, surprising elsewhere). Follow-up: many MovieLens ratings share the same second (bulk imports of old ratings when a user joins), so huge "sessions" are an artefact of how the data was collected. In product logs you would also cap session length or split on app background events.

**Complexity:** Two window passes per user over 100,836 ratings, then group bys.

**Common mistakes:** Running SUM without PARTITION BY user (session numbers leak across users). Comparing a gap in days with 30 (wrong units). Forgetting the first rating of each user: its LAG is NULL, and `NULL > 30` is not true, so without the IS NULL test it would not start a session.

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals)

</details>

---
### Q4. Long breaks and comebacks (review)

*medium, about 5 min*

Shopping app. A **comeback** is a pair of consecutive active days of the same user that are **more
than 14 days apart** (the user was away and came back). Per `platform` return `comebacks`, `users_with_comeback`,
`avg_gap_days` (1 decimal) and `max_gap`.

Example: active on Jan 2, Jan 20, Jan 22 gives one comeback with a gap of 18 days.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: gaps (not islands): LEAD gives each active day the next active day of the same user; filter where the difference is > 14.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `LEAD(activity_date) OVER (PARTITION BY user_id ORDER BY activity_date) AS next_date`.
2. Gap = `julianday(next_date) - julianday(activity_date)`; keep > 14; group by platform.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH nxt AS (
    SELECT d.user_id, u.platform, d.activity_date,
           LEAD(d.activity_date) OVER (PARTITION BY d.user_id ORDER BY d.activity_date) AS next_date
    FROM daily_activity d
    JOIN app_users u ON u.user_id = d.user_id
)
SELECT platform,
       COUNT(*)                AS comebacks,
       COUNT(DISTINCT user_id) AS users_with_comeback,
       ROUND(AVG(julianday(next_date) - julianday(activity_date)), 1) AS avg_gap_days,
       MAX(julianday(next_date) - julianday(activity_date))           AS max_gap
FROM nxt
WHERE julianday(next_date) - julianday(activity_date) > 14
GROUP BY platform
ORDER BY platform
```

Result:

| platform | comebacks | users_with_comeback | avg_gap_days | max_gap |
|---|---|---|---|---|
| android | 227 | 201 | 30.5 | 76.0 |
| ios | 181 | 159 | 31.7 | 79.0 |
| web | 71 | 60 | 27.2 | 69.0 |

**Why:** Gaps are the space between islands; LEAD exposes each gap on one row. The last active day of every user has a NULL next_date, and the WHERE comparison with NULL is not true, so it drops out automatically. Gaps average about a month. Users who left and never came back are not comebacks; a churn analysis would look at the gap from the last active day to the end of the data instead.

**Complexity:** One window sort over 17,223 rows.

**Common mistakes:** LAG and LEAD mixed up (both work if used consistently). Forgetting PARTITION BY user. Counting users instead of gaps when the question asks for comebacks.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands)

</details>

---
### Q5. How much activity comes from streaks? (review)

*medium, about 5 min*

Shopping app. Label each active day with the length of the streak it belongs to. Per `platform` return
`active_days`, `days_in_streak3` (active days that belong to a streak of 3 or more days) and `pct_in_streak3`
(1 decimal).

Example: a user with streaks of 1, 4 and 2 days has 7 active days, 4 of them in a 3+ streak (57.1%).

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: islands, but instead of collapsing each island to one row, attach its size to every row with `COUNT(*) OVER (PARTITION BY user_id, island)`; then a conditional ratio.

</details>

<details><summary><b>Hint 2</b></summary>

1. grp CTE as in Q1.
2. `COUNT(*) OVER (PARTITION BY user_id, island) AS streak_len`.
3. Join platform; `SUM(CASE WHEN streak_len >= 3 ...) / COUNT(*)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH grp AS (
    SELECT user_id, activity_date,
           julianday(activity_date)
             - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) AS island
    FROM daily_activity
), sized AS (
    SELECT user_id, COUNT(*) OVER (PARTITION BY user_id, island) AS streak_len
    FROM grp
)
SELECT u.platform,
       COUNT(*) AS active_days,
       SUM(CASE WHEN s.streak_len >= 3 THEN 1 ELSE 0 END) AS days_in_streak3,
       ROUND(100.0 * SUM(CASE WHEN s.streak_len >= 3 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_in_streak3
FROM sized s
JOIN app_users u ON u.user_id = s.user_id
GROUP BY u.platform
ORDER BY u.platform
```

Result:

| platform | active_days | days_in_streak3 | pct_in_streak3 |
|---|---|---|---|
| android | 7805 | 5608 | 71.9 |
| ios | 7427 | 5902 | 79.5 |
| web | 1991 | 1336 | 67.1 |

**Why:** A window COUNT over the island keeps every day as a row but tells it how big its island is, so the ratio is over days (the denominator the question asks for). iOS has the most streaky usage (79.5% of its active days are in 3+ day streaks), web the least (67.1%). Habit-forming products track this kind of metric (share of usage from habitual users).

**Complexity:** Two window passes, one join, one group by.

**Common mistakes:** Collapsing islands first and then dividing islands by days (mixed units). Computing the ratio per user and averaging (average of ratios, see day 18).

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_19/3_stats.ipynb)